# Phase D — GAN (Stufe 2)

Derselbe **U-Net-Generator** wie in Phase C, jetzt **adversarial feingetunt** gegen einen
**PatchGAN-Diskriminator** (Spectral Norm). Ziel: das rekonstruierte HF von *verwaschen*
(Regression) zu *scharf* (GAN) bringen. Das ist die zentrale Pointe — und die Ablation
**Regression vs. GAN** ist genau der Schalter `λ_adv = 0` vs. `> 0`.

Training läuft auf Kaggle (`kaggle_train_gan.ipynb`); die Ergebnis-Zellen unten greifen
automatisch, sobald `gan_*/generator.weights.h5` unter `BWE_CKPT_ROOT` liegt. Importiert aus
`bwe/` — nicht self-contained.

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import glob
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import tensorflow as tf

from bwe import config as cfg
from bwe.models.generator import Generator, build_unet
from bwe.models.discriminator import build_discriminator
from bwe.data.loaders import load_demo
from bwe.infer.reconstruct import model_from_fullband, baseline_from_fullband
from bwe.eval import metrics as M
from bwe.eval import plots as P

print(cfg.summary())
print(f"\nGAN: λ_adv={cfg.LAMBDA_ADV}  λ_fm={cfg.LAMBDA_FM}  D-Kanäle={cfg.DISC_CHANNELS}  "
      f"Spectral-Norm={cfg.DISC_USE_SPECTRAL_NORM}")

## Architektur

**Generator** = der U-Net aus Phase C (warm-gestartet). **Diskriminator** = PatchGAN: 3×
`Conv2D`-s2 (64→128→256) mit `LeakyReLU(0.2)` und **Spectral Norm**, finale 1-Kanal-Conv →
Patch-Gitter (lokales Urteil je Spektrogramm-Region). Eingang = volles Spektrogramm `[B,512,T,2]`;
weil das LF vor dem D gesplict wird (in echt wie fake identisch), zielt das Urteil aufs HF.

In [ ]:
print("=== Generator (U-Net) ===")
build_unet().summary()
print("\n=== Diskriminator (PatchGAN) ===")
build_discriminator().summary()

## Trainingsprinzip (zweiphasig)

1. **Warm-Start**: Generator-Gewichte aus der Regression — hebt das Startniveau, statt blind zu beginnen.
2. **Diskriminator-Vorlauf**: einige hundert Steps nur D (G eingefroren), damit sein Urteil schon trägt.
3. **Adversariale Phase** (pro Step zwei `GradientTape`-Blöcke):
   - **D**: `bce(1, echt) + bce(0, fake)` — echtes vs. generiertes HF trennen.
   - **G**: `L_recon + λ_adv·bce(1, fake) + λ_fm·L_fm` — Rekonstruktions-Anker **plus** den D täuschen
     **plus** Feature-Matching (D-Zwischenaktivierungen angleichen, stabilisiert).

`g_loss` ist **kein** Qualitätsmaß (D bewegt sich) → Modellwahl per **Val-LSD-HF + Reinhören**.

## Lernkurven (nach dem Training)

In [ ]:
logs = sorted(glob.glob(str(cfg.CKPT_ROOT / "**" / "log.csv"), recursive=True))
logs = [l for l in logs if "gan" in l.lower()]
if logs:
    import pandas as pd
    df = pd.read_csv(logs[-1])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(df["epoch"], df["d_loss"], label="d_loss")
    ax[0].plot(df["epoch"], df["g_loss"], label="g_loss")
    if "recon" in df: ax[0].plot(df["epoch"], df["recon"], label="recon", ls="--")
    ax[0].set_title("GAN-Losses (kein Qualitätsmaß!)"); ax[0].set_xlabel("Epoche"); ax[0].legend()
    if "val_lsd_hf" in df:
        ax[1].plot(df["epoch"], df["val_lsd_hf"]); ax[1].set_title("Val-LSD-HF [dB] (Qualitätsmaß)")
        ax[1].set_xlabel("Epoche")
    plt.tight_layout(); plt.show()
else:
    print("Noch keine GAN-log.csv — zuerst auf Kaggle trainieren (Schritt 16).")

## Ablation: Copy-Up vs. Regression vs. GAN vs. Original (nach dem Training)

Die Kern-Gegenüberstellung. Erwartung (Leitfaden §5.5): das GAN senkt die **LSD-HF** und macht das
HF hör- und sichtbar schärfer, **verschlechtert** dabei aber oft die **SI-SDR** (es erfindet
plausibles, aber nicht phasentreues HF) — die „Metrik vs. Ohr"-Pointe.

In [ ]:
allw = sorted(glob.glob(str(cfg.CKPT_ROOT / "**" / "generator.weights.h5"), recursive=True))
gan_w = [w for w in allw if "gan" in w.lower()]
reg_w = [w for w in allw if "reg" in w.lower()]
if not gan_w:
    print("Noch kein trainiertes GAN. Lege gan_*/generator.weights.h5 unter", cfg.CKPT_ROOT,
          "ab (Kaggle-Output) und führe diese Zelle erneut aus.")
else:
    def load_gen(path):
        g = Generator(); g(tf.zeros([1, cfg.N_BINS_NET, cfg.SEG_FRAMES, cfg.N_INPUT_CHANNELS]))
        g.load_weights(path); return g

    g_gan = load_gen(gan_w[-1]); print("GAN-Gewichte:", gan_w[-1])
    g_reg = load_gen(reg_w[-1]) if reg_w else None

    name, target = load_demo("valid", 0, seconds=6.0, offset=30.0)
    gan, inp = model_from_fullband(g_gan, target)
    cu, _ = baseline_from_fullband(target)
    reg = model_from_fullband(g_reg, target)[0] if g_reg is not None else None

    left = reg if reg is not None else inp
    P.spectro_triple(left, gan, target,
                     titles=("Regression" if reg is not None else "Bandbegrenzt", "GAN", "Original"))
    plt.show()

    print(f"{'':12s}{'LSD-HF [dB]':>14s}{'SI-SDR [dB]':>14s}")
    rows = [("Copy-Up", cu)]
    if reg is not None: rows.append(("Regression", reg))
    rows.append(("GAN", gan))
    for lbl, w in rows:
        print(f"{lbl:12s}{M.lsd_hf(w, target):14.2f}{M.si_sdr(w, target):14.2f}")

    print("\nAnhören:")
    print("Bandbegrenzt:"); display(Audio(inp.numpy(), rate=cfg.SR))
    if reg is not None: print("Regression:"); display(Audio(reg.numpy(), rate=cfg.SR))
    print("GAN:"); display(Audio(gan.numpy(), rate=cfg.SR))
    print("Original:"); display(Audio(target, rate=cfg.SR))

## Fazit

Das GAN ersetzt das *verwaschene* HF der Regression durch *scharfe*, plausible Struktur — die
LSD-HF sinkt weiter, der Höreindruck gewinnt. Der Preis ist oft eine **schlechtere SI-SDR**:
scharfes, aber nicht phasentreues HF wird von der signalbezogenen Metrik bestraft, vom Ohr
aber bevorzugt. Diese Divergenz ist die eigentliche Einsicht des Projekts — und der Grund,
warum BWE adversarial und nicht rein per Regression gelöst wird.